In [8]:
import json
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from matplotlib.font_manager import FontProperties
from matplotlib.textpath import TextPath
from scipy.stats import gaussian_kde

# ---- Style (matches train_figs) -------------------------------------
STYLE = {
    "figsize": (10, 6),
    "font_size": 18,
    "linewidth": 2.5,
    "grid": True,                  # GRIDLINES
    "dpi": 150,
    "legend_fontsize": 16,
    "legend_linewidth": 3.0,       # line width in the legend only
    "stat_fontsize": 17,           # per-run stat labels beside the curves

    # ---- Grouping (as in train_figs) ---------------------------------
    "runs_per_row": 2,             # GUI: run tickboxes per grid row
    # Runs whose name starts with one of these prefixes always form their
    # own, completely separate groups (under every grouping).
    "separate_prefixes": ["4800", "9200"],
}

plt.rcParams.update({
    # "font.family": "STIXGeneral",
    "mathtext.fontset": "stix",
})


def style_axes(ax):
    """Grid + full spines, as in train_figs."""
    if STYLE["grid"]:
        ax.grid(True, alpha=0.3)


def style_legend(leg):
    """Thicker lines in the legend, as in train_figs."""
    for line in leg.get_lines():
        line.set_linewidth(STYLE["legend_linewidth"])


import matplotlib.patheffects as pe
import matplotlib.colors as mcolors

def darken(color, factor=0.45):
    r, g, b = mcolors.to_rgb(color)
    return (r * factor, g * factor, b * factor)

def annotate_peak_stats(ax, grid, entries, all_curves=None, fontsize=None):
    """Write each run's stat (e.g. '+4.1%') in free space beside its curve.

    entries:    [(text, density_array, colour), ...] on the shared x grid.
    all_curves: every plotted density (labels also dodge runs w/o stats).

    Each label is anchored to its own curve and slides along it to the
    clearest nearby spot: candidates are scored on (a) no curve poking
    into the label box, (b) no overlap with labels already placed,
    (c) staying close to the peak, (d) staying close to the curve.
    Clean spots always beat violating ones. If the curve is boxed in on
    all sides (dense plots), the label floats up into open space and a
    thin leader line ties it back to its curve.
    """
    if not entries:
        return
    ax.autoscale_view(scalex=False)          # make sure y-limits are final
    fontsize = fontsize or STYLE.get("stat_fontsize", STYLE["legend_fontsize"])

    # measure each label's true rendered size, in points
    try:
        ren = ax.figure.canvas.get_renderer()
    except Exception:
        ren = None                           # backend without get_renderer
    px2pt, sizes = 72.0 / ax.figure.dpi, []
    fp = FontProperties(weight="bold", size=fontsize)
    for t, _, _ in entries:
        if ren is not None:
            probe = ax.text(0, 0, t, fontsize=fontsize, fontweight="bold",
                            alpha=0)
            bb = probe.get_window_extent(ren)
            probe.remove()
            sizes.append((bb.width * px2pt, bb.height * px2pt))
        else:                                # conservative estimate
            e = TextPath((0, 0), t, size=fontsize, prop=fp).get_extents()
            sizes.append((e.width + 4, 1.3 * fontsize))

    curves = all_curves if all_curves else [d for _, d, _ in entries]
    env = np.max(np.vstack(curves), axis=0)  # upper envelope of all curves

    # points -> data units, + just enough headroom to fit a label above
    # the tallest peak (solved for the post-rescale axis height)
    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    fig_w, fig_h = ax.figure.get_size_inches()
    ax_w_pt = ax.get_position().width * fig_w * 72
    ax_h_pt = ax.get_position().height * fig_h * 72
    need_pt = 5 + max(h for _, h in sizes) + 6       # gap + text + margin
    k = min(need_pt / ax_h_pt, 0.45)
    y1 = max(y1, (env.max() - k * y0) / (1 - k))
    ax.set_ylim(y0, y1)
    ux = (x1 - x0) / ax_w_pt                 # data units per point
    uy = (y1 - y0) / ax_h_pt

    LEAD = 15                                # lifts beyond this get a leader
    step = grid[1] - grid[0]
    placed = []                              # reserved boxes: (l, r, b, t)
    order = sorted(range(len(entries)), key=lambda i: -entries[i][1].max())
    for i in order:
        text, dens, colour = entries[i]
        others = [c for c in curves if c is not dens]
        oenv = np.max(np.vstack(others), axis=0) if others else None
        w, h = sizes[i][0] * ux, (sizes[i][1] + 2) * uy
        gap, half = 5 * uy, w / 2 + 3 * ux
        hw = int(np.ceil(half / step))       # half box width in grid steps
        pk = int(np.argmax(dens))
        best, best_cost = (pk, dens[pk] + gap, 0.0), None
        for j in range(hw, len(grid) - hw, 2):
            crest = dens[j - hw:j + hw + 1].max()   # own curve stays below
            for lift in (0, 6, 12, 20, 30, 45, 65, 90):
                yb = crest + gap + lift * uy
                crowd = 0.0 if oenv is None else \
                    max(0.0, oenv[j - hw:j + hw + 1].max() - yb)
                l, r, t_ = grid[j] - half, grid[j] + half, yb + h
                olap = sum(max(0.0, min(r, pr) - max(l, pl)) *
                           max(0.0, min(t_, pt) - max(yb, pb))
                           for pl, pr, pb, pt in placed)
                if lift > LEAD:              # leader strip must be clear too
                    ll, lr = grid[j] - 3 * ux, grid[j] + 3 * ux
                    olap += sum(max(0.0, min(lr, pr) - max(ll, pl)) *
                                max(0.0, min(yb, pt) - max(dens[j], pb))
                                for pl, pr, pb, pt in placed)
                over = max(0.0, t_ - y1)     # pokes past the top limit
                bad = crowd > 3 * uy or olap > 0 or over > 0
                cost = (0.7 * abs(grid[j] - grid[pk]) / (x1 - x0)
                        + 0.03 * lift + (5.0 if bad else 0.0)
                        + 8 * crowd / (y1 - y0) + 30 * olap / (w * h)
                        + 50 * over / (y1 - y0))
                if best_cost is None or cost < best_cost:
                    best_cost, best = cost, (j, yb, lift)
        j, yb, lift = best
        placed.append((grid[j] - half, grid[j] + half, yb, yb + h))
        if lift > LEAD:                      # thin leader back to the curve
            ax.plot([grid[j], grid[j]], [dens[j] + 3 * uy, yb - 2 * uy],
                    color=colour, lw=1.2, alpha=0.65, zorder=4,
                    solid_capstyle="round")
            placed.append((grid[j] - 3 * ux, grid[j] + 3 * ux,
                           dens[j], yb))
                    
            
        ax.text(
            grid[j],
            yb,
            text,
            ha="center",
            va="bottom",
            fontsize=fontsize,
            fontweight=600,          # semibold
            color=colour,
            zorder=10,
            clip_on=False,

            path_effects=[
                pe.Stroke(
                    linewidth=3,
                    foreground="white"
                ),
                pe.Normal(),
            ]
        )

# ---- Grouping / legend ordering (ported from train_figs) -------------
def parse_attrs(run_name, prefixes=None):
    """Pull grouping attributes (model family, ε settings, lr, seed, gt tag,
    batch prefix) out of a run name like 'r2_qwen__low=0.1_high=inf_s0'.

    gt:     True if 'gt' or 'GT' appears anywhere in the name.
    prefix: the matching STYLE["separate_prefixes"] entry (e.g. '4800') if
            the name starts with one, else None.
    """
    if prefixes is None:
        prefixes = STYLE.get("separate_prefixes", ())

    def grab(pattern):
        m = re.search(pattern, run_name)
        return m.group(1) if m else None

    m = re.search(r"(olmo|llama|qwen)", run_name)
    ds = re.search(r"(countdown|dapo|mbpp)", run_name)
    return {
        "model":  m.group(1) if m else "other",
        "ds":     ds.group(1) if ds else "gsm8k",
        "gt":     False, # "gt" in run_name or "GT" in run_name,
        "prefix": next((p for p in prefixes if run_name.startswith(p)), None),
        "low":    grab(r"low=([\d.]+|inf)"),
        "high":   grab(r"high=([\d.]+|inf)"),
        "lr":     grab(r"lr=([\d.eE+-]+)"),
        "seed":   grab(r"_s(\d+)"),
    }


def _as_float(text, default):
    """Text-box value -> float; blank or unparseable falls back to default."""
    try:
        return float(text)
    except (TypeError, ValueError):
        return default


def _cache_key(label):
    """Key under which a run's colour is stored in color_cache.json (shared
    with train_figs). Eval labels carry a trailing '_steps=<n>' that the
    training run names don't, so it's stripped to make the two line up."""
    return re.sub(r"_steps=\d+$", "", label)


def order_legend(handles, labels, names, order):
    """Reorder legend entries so matching labels/run names come first."""
    if not order:
        return handles, labels
    remaining = list(range(len(labels)))
    ranked = []
    for token in order:
        for j in remaining:
            if token in labels[j] or token in names[j]:
                ranked.append(j)
                remaining.remove(j)
                break
    ranked.extend(remaining)          # unmatched keep their original order
    return [handles[j] for j in ranked], [labels[j] for j in ranked]


def _num(s):
    """'0.28' -> 0.28, 'inf' -> inf, None/other -> inf (sorts last)."""
    try:
        return float(s)
    except (TypeError, ValueError):
        return float("inf")


In [9]:

METRICS = ["avg_logprob", "logprob_per_char", "mean_entropy", "num_tokens"]

def ds_lookup(ds_name):
    if "gsm8k" in ds_name:
        return "gsm8k"
    if "AIME" in ds_name or "dapo" in ds_name:
        return "dapo"
    if "wordle" in ds_name:
        return "wordle"
    if "countdown" in ds_name:
        return "countdown"
    if "mbpp" in ds_name:
        return "mbpp"
    return ds_name

def map_name(e):
    if e["label"] == "olmo" or e["label"] == "qwen" or e["label"] == "llama" or "r2" in e["label"]:  
        e["label"] = e["label"] + "-" + ds_lookup(e["dataset"])
    return e

def load_results(path="results/results_output_dist.json"):
    """Load the results file and return a {label: entry} dict."""
    with open(path) as f:
        results = json.load(f)

    return {map_name(e)["label"]: e for e in results}

def short_label(label):
    """Compress long run names, e.g.
    'lr_run_qwen_lr=1e-6_low=1.0_high=0.1_s0_steps=2400' -> 'qwen lr=1e-6'."""
    m = re.search(r"(olmo|llama|qwen)", label)
    model = m.group(1) if m else label
    lr = re.search(r"lr=([\d.e-]+)", label)
    lo = re.search(r"low=([\d.]+|inf)", label)
    hi = re.search(r"high=([\d.]+|inf)", label)
    bits = [model]
    if lr: bits.append(f"lr={lr.group(1)}")
    if lo and hi: bits.append(f"[{lo.group(1)},{hi.group(1)}]")
    return " ".join(bits) if len(bits) > 1 else label

def plot_metric(results, metric="avg_logprob", labels=None, ax=None,
                show_greedy=False, clip=(1, 99), bw=0.25, lw=None,
                title=None):
    """Overlay smoothed (KDE) output-distribution curves for one metric.

    clip: percentile range (across all shown runs) used for the x-axis,
          so a few outliers don't flatten everything into a spike.
    bw:   KDE bandwidth factor; raise for smoother, lower for more detail.
    """
    if labels is None:
        labels = list(results.keys())
    plt.rc("font", size=STYLE["font_size"])
    if ax is None:
        _, ax = plt.subplots(figsize=STYLE["figsize"])
    lw = lw or STYLE["linewidth"]

    print(f"Metric: {metric}")

    if metric == "avg_logprob":

        for l in labels:
            results[l]["distributions"][metric] *= -1

    all_vals = np.concatenate(
        [np.asarray(results[l]["distributions"][metric], float) for l in labels])
    lo, hi = np.percentile(all_vals, clip)
    pad = 0.05 * (hi - lo)
    grid = np.linspace(lo - pad, hi + pad, 400)

    for label in labels:
        e = results[label]
        vals = np.asarray(e["distributions"][metric], float)

        kde = gaussian_kde(vals, bw_method=bw)
        (line,) = ax.plot(grid, kde(grid), label=short_label(label), lw=lw)
        if show_greedy:
            g = e["greedy_ref"].get(metric)
            if g is not None and lo - pad <= g <= hi + pad:
                ax.axvline(g, color=line.get_color(), ls="--", lw=1, alpha=0.5)

    ax.set_xlim(lo - pad, hi + pad)
    l = "Mean token log-probability" if metric == "avg_logprob" else metric
    ax.set_xlabel(l)
    ax.set_ylabel("Density")
    if title:
        ax.set_title(title)
    style_axes(ax)
    leg = ax.legend(fontsize=STYLE["legend_fontsize"],
                    ncol=len(labels), loc="lower center",
                    bbox_to_anchor=(0.5, 1.01))
    style_legend(leg)
    return ax

def plot_all_metrics(results, labels=None, clip=(1, 99), bw=0.25,
                     suptitle=None):
    """2x2 grid: one KDE panel per metric, shared legend above."""
    plt.rc("font", size=STYLE["font_size"])
    w, h = STYLE["figsize"]
    fig, axes = plt.subplots(2, 2, figsize=(1.6 * w, 1.6 * h))
    for metric, ax in zip(METRICS, axes.ravel()):
        plot_metric(results, metric, labels=labels, ax=ax, clip=clip, bw=bw)
        ax.get_legend().remove()
    handles, leg_labels = axes.ravel()[0].get_legend_handles_labels()
    leg = fig.legend(handles, leg_labels, loc="lower center",
                     ncol=min(4, len(leg_labels)),
                     fontsize=STYLE["legend_fontsize"],
                     bbox_to_anchor=(0.5, 1.0))
    style_legend(leg)
    if suptitle:
        fig.suptitle(suptitle)
    fig.tight_layout()
    return fig

# Example usage:
# results = load_results("results_output_dist.json")
# plot_all_metrics(results, labels=[l for l in results if "lr_run_qwen" in l])
# plot_metric(results, "mean_entropy", labels=["qwen", "run0_qwen_lr=5e-7_low=0.2_high=0.2_s0_steps=2400"])
# plt.show()


In [10]:


def load_entropy_results(path="results/results_entropy_gsm8k.json"):
    """Load the entropy results file and return a {label: entry} dict."""
    with open(path) as f:
        results = json.load(f)
    return {e["label"]: e for e in results}

def _short_label(label):
    m = re.search(r"(olmo|llama|qwen)", label)
    model = m.group(1) if m else label
    lr = re.search(r"lr=([\d.e-]+)", label)
    lo = re.search(r"low=([\d.]+|inf)", label)
    hi = re.search(r"high=([\d.]+|inf)", label)
    bits = [model]
    if lr: bits.append(f"lr={lr.group(1)}")
    if lo and hi: bits.append(f"[{lo.group(1)},{hi.group(1)}]")
    return " ".join(bits) if len(bits) > 1 else label

def plot_entropy_curves(results, labels=None, ax=None, clip=(1, 99),
                        bw=0.25, lw=None, show_mean=True,
                        token_weighted=False, title=None):
    """Overlay smoothed per-generation mean-entropy distributions.

    Pulls `mean_entropy` from each run's `per_generation` list
    (num_prompts x num_generations values, e.g. 1024 per run).

    show_mean:      dashed vertical line at each run's mean.
    token_weighted: weight each generation's KDE contribution by its
                    num_tokens, so the curve reflects per-token rather
                    than per-generation entropy. The dashed line then
                    uses the file's token_mean_entropy_nats.
    """
    if labels is None:
        labels = list(results.keys())
    plt.rc("font", size=STYLE["font_size"])
    if ax is None:
        _, ax = plt.subplots(figsize=STYLE["figsize"])
    lw = lw or STYLE["linewidth"]

    per_run = {}
    for l in labels:
        gens = results[l]["per_generation"]
        vals = np.array([g["mean_entropy"] for g in gens], float)
        wts = np.array([g["num_tokens"] for g in gens], float)
        per_run[l] = (vals, wts)

    all_vals = np.concatenate([v for v, _ in per_run.values()])
    lo, hi = np.percentile(all_vals, clip)
    pad = 0.05 * (hi - lo)
    grid = np.linspace(lo - pad, hi + pad, 400)

    for label in labels:
        vals, wts = per_run[label]
        kde = gaussian_kde(vals, bw_method=bw,
                           weights=wts if token_weighted else None)
        (line,) = ax.plot(grid, kde(grid), label=_short_label(label), lw=lw)
        if show_mean:
            m = (results[label]["token_mean_entropy_nats"] if token_weighted
                 else results[label]["per_generation_mean_nats"])
            if lo - pad <= m <= hi + pad:
                ax.axvline(m, color=line.get_color(), ls="--", lw=1, alpha=0.5)

    ax.set_xlim(lo - pad, hi + pad)
    ax.set_xlabel("mean entropy (nats)")
    ax.set_ylabel("Density")
    if title:
        ax.set_title(title)
    style_axes(ax)
    leg = ax.legend(fontsize=STYLE["legend_fontsize"],
                    ncol=len(labels), loc="lower center",
                    bbox_to_anchor=(0.5, 1.01))
    style_legend(leg)
    return ax


In [11]:
"""Notebook GUI for building output-distribution / entropy figures.

Usage in a notebook cell:

    from plot_output_dist import load_results
    from plot_entropy_dist import load_entropy_results
    from figure_gui import FigureGUI

    gui = FigureGUI(
        output_dist=load_results("results_output_dist.json"),
        entropy=load_entropy_results("results_entropy_gsm8k.json"),
    )
    gui.show()

Requires: pip install ipywidgets  (then restart the kernel if it was missing)
"""
import re
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from scipy.stats import gaussian_kde
import ipywidgets as w
from IPython.display import display

_OUTPUT_DIST_METRICS = ["avg_logprob", "logprob_per_char",
                        "mean_entropy", "num_tokens"]

def _short_label(label):
    m = re.search(r"(olmo|llama|qwen)", label)
    model = m.group(1) if m else label
    lr = re.search(r"lr=([\d.e-]+)", label)
    lo = re.search(r"low=([\d.]+|inf)", label)
    hi = re.search(r"high=([\d.]+|inf)", label)
    bits = [model]
    if lr: bits.append(f"lr={lr.group(1)}")
    if lo and hi: bits.append(f"[{lo.group(1)},{hi.group(1)}]")
    return " ".join(bits) if len(bits) > 1 else label



def _eps_tex(lo, hi):
    lo = r"\infty" if lo == "inf" else lo
    hi = r"\infty" if hi == "inf" else hi
    return (rf"\varepsilon_{{\mathrm{{low}}}}={lo}," + "\:" + 
            rf"\varepsilon_{{\mathrm{{high}}}}={hi}")


def short_label(run_name):
    """Default legend label, e.g. 'r2_qwen__low=0.1_high=inf_s0' ->
    'ε_low=0.1, ε_high=∞' rendered in mathtext, with the batch prefix,
    'gt' and 'η=<lr>' prepended when present."""
    a = parse_attrs(run_name)
    bits = []
    if a["prefix"]:
        bits.append("\mathrm{" +a["prefix"] + "}")
    if a["gt"]:
        bits.append("\mathrm{GT}")
    if a["lr"]:
        bits.append(rf"\eta={a['lr']}")
    if a["low"] and a["high"]:
        bits.append(_eps_tex(a["low"], a["high"]))
    
    if bits:
        return "$" + " \: ".join(bits) + "$" 
    else:

        return run_name # f"{run_name}-{a['ds']}"

class FigureGUI:
    # group name <- run attributes (see parse_attrs); runs with a batch
    # prefix (4800/9200/...) get "<prefix> · " prepended in _current_groups,
    # so they never share a group with unprefixed runs.
    GROUPINGS = {
        "model family":  lambda a: a["model"] + (" gt" if a["gt"] else ""),
        "ε (low, high)": lambda a: f"low={a['low'] or '?'}, high={a['high'] or '?'}",
        "ε low":         lambda a: f"low={a['low'] or '?'}",
        "ε high":        lambda a: f"high={a['high'] or '?'}",
        "lr":            lambda a: f"lr={a['lr'] or '?'}",
        "none":          lambda a: "all runs",
    }

    def __init__(self, output_dist=None, entropy=None, default_selected=None):
        """output_dist / entropy: {label: entry} dicts from the loaders.
        default_selected: iterable of labels ticked at start (default: none)."""
        if output_dist is None and entropy is None:
            raise ValueError("pass at least one of output_dist / entropy")
        self.output_dist = output_dist or {}
        self.entropy = entropy or {}


        labels = list(dict.fromkeys(list(self.output_dist) + list(self.entropy)))
        self.attrs = {l: parse_attrs(l) for l in labels}
        # stable base order: batch prefix, model, gt, then ε low, ε high, lr, name
        self.labels = sorted(labels, key=lambda n: (
            self.attrs[n]["prefix"] or "", self.attrs[n]["model"],
            self.attrs[n]["gt"], _num(self.attrs[n]["low"]),
            _num(self.attrs[n]["high"]), _num(self.attrs[n]["lr"]), n))
        default_selected = set(default_selected or [])

        self._guard = False          # silences observers during bulk updates
        self._color_guard = False    # silences colour save during bulk updates
        self._child_handlers = {}    # label -> tick observer (for regroup)

        cmap = plt.get_cmap("tab20")
        def hexcolor(i):
            r, g, b, _ = cmap(i % 20)
            return "#{:02x}{:02x}{:02x}".format(int(r*255), int(g*255), int(b*255))

        # --- Load Color Cache (shared with train_figs) ---
        self.cache_path = Path("color_cache.json")
        self.color_cache = {}
        if self.cache_path.exists():
            try:
                with open(self.cache_path, "r") as f:
                    self.color_cache = json.load(f)
            except Exception as e:
                print(f"Warning: Failed to load color cache: {e}")

        # --- one fixed row per run; regrouping only re-orders these ---
        self.rows, self.row_boxes = {}, {}
        for i, label in enumerate(self.labels):
            tick = w.Checkbox(value=label in default_selected, indent=False,
                              layout=w.Layout(width="24px"))
            name = w.Text(value=short_label(label),
                          layout=w.Layout(width="220px"))
            # Use cached color if available, otherwise default
            key = _cache_key(label)
            default_color = self.color_cache.get(key, hexcolor(i))
            color = w.ColorPicker(concise=True, value=default_color,
                                  layout=w.Layout(width="36px"))

            # Add to cache if it's new and bind observer
            self.color_cache[key] = default_color
            color.observe(lambda change, k=key: self._on_color_change(change, k),
                          names="value")
            stat = w.Text(value="", placeholder="e.g. +4.1%",
                          layout=w.Layout(width="90px"))
            full = w.Label(label, layout=w.Layout(width="auto",
                                                  margin="0 0 0 6px"),
                           style={"font_size": "11px"})
            self.rows[label] = (tick, name, color, stat)
            self.row_boxes[label] = w.HBox([tick, color,name, stat, full])

        self._save_color_cache()  # Save initial state (captures any new runs)

        sources = []
        # _OUTPUT_DIST_METRICS.remove("avg_logprob")
        _OUTPUT_DIST_METRICS.remove("logprob_per_char")

        if self.output_dist:
            sources += [f"output_dist: {m}" for m in _OUTPUT_DIST_METRICS]
        if self.entropy:
            sources += ["entropy: per-generation", "entropy: token-weighted"]

        # --- global controls ---
        self.source = w.Dropdown(options=sources, description="metric",
                                 layout=w.Layout(width="320px"))
        self.group_by = w.Dropdown(options=list(self.GROUPINGS),
                                   value="model family", description="group by",
                                   layout=w.Layout(width="260px"))
        self.bw = w.FloatSlider(value=0.25, min=0.05, max=1.0, step=0.05,
                                description="bandwidth", continuous_update=False)
        self.clip_lo = w.FloatSlider(value=1, min=0, max=10, step=0.5,
                                     description="clip low %", continuous_update=False)
        self.clip_hi = w.FloatSlider(value=99, min=90, max=100, step=0.5,
                                     description="clip high %", continuous_update=False)
        self.show_ref = w.Checkbox(value=False, description="dashed ref lines (greedy / mean)",
                                   indent=False)
        self.title = w.Text(value="", description="title",
                            placeholder="(auto)", layout=w.Layout(width="400px"))
        self.legend_order = w.Text(value="Q, L, O", description="legend order",
                                   placeholder="e.g. 5e-7, 1e-6, 2e-6",
                                   layout=w.Layout(width="300px"))
        self.stat_mode = w.Dropdown(options=["off", "in legend",
                                             "at curve peak"],
                                    value="off", description="show stat",
                                    layout=w.Layout(width="240px"))
        self.xlabel = w.Text(value="Mean token entropy", description="x label",
                             placeholder="(auto)", layout=w.Layout(width="300px"))
        self.ylabel = w.Text(value="Density", description="y label",
                             layout=w.Layout(width="300px"))
        # crop the x axis by hand; blank = use the clip percentiles
        self.xmin = w.Text(value="", description="x min", placeholder="(auto)",
                           layout=w.Layout(width="180px"))
        self.xmax = w.Text(value="", description="x max", placeholder="(auto)",
                           layout=w.Layout(width="180px"))
        self.fname = w.Text(value="#c3eval/figure.pdf", description="save as",
                            layout=w.Layout(width="300px"))
        plot_btn = w.Button(description="Plot", button_style="primary")
        save_btn = w.Button(description="Save")
        sel_all = w.Button(description="select all", layout=w.Layout(width="90px"))
        sel_none = w.Button(description="clear", layout=w.Layout(width="90px"))
        colour_btn = w.Button(description="colour by group",
                              layout=w.Layout(width="130px"))
        self.out = w.Output()
        self._fig = None

        plot_btn.on_click(lambda _: self._plot())
        save_btn.on_click(lambda _: self._save())
        sel_all.on_click(lambda _: self._set_all(True))
        sel_none.on_click(lambda _: self._set_all(False))
        colour_btn.on_click(lambda _: self._colour_by_group())
        self.group_by.observe(lambda _: self._regroup(), names="value")

        header = w.HTML("<b>runs</b> &nbsp; (tick | colour | legend label | "
                        "stat | source label)")
        self.runs_box = w.VBox()
        self._regroup()

        self.ui = w.VBox([
            w.HBox([self.source, self.group_by, sel_all, sel_none, colour_btn]),
            header, self.runs_box,
            w.HBox([self.bw, self.clip_lo, self.clip_hi]),
            w.HBox([self.show_ref, self.title, self.legend_order]),
            w.HBox([self.xlabel, self.ylabel, self.stat_mode]),
            w.HBox([self.xmin, self.xmax]),            
            w.HBox([plot_btn, save_btn, self.fname]),
            self.out,
        ])

    def show(self):
        display(self.ui)

    def _set_all(self, val):
        for tick, *_ in self.rows.values():
            tick.value = val

    # --- color caching -----------------------------------------------------
    def _on_color_change(self, change, key):
        """Update cache when user manually changes a colour."""
        self.color_cache[key] = change["new"]
        if not self._color_guard:
            self._save_color_cache()

    def _save_color_cache(self):
        """Write the current colours to disk."""
        try:
            with open(self.cache_path, "w") as f:
                json.dump(self.color_cache, f, indent=2)
        except Exception as e:
            print(f"Failed to save color cache: {e}")

    # --- grouping ----------------------------------------------------------
    def _current_groups(self):
        """{group name: [labels]} under the selected grouping, in the stable
        base order (so both groups and members come out sorted). Runs with a
        batch prefix (separate_prefixes) get their own '<prefix> · <group>'
        groups — never mixed with unprefixed runs."""
        keyfn = self.GROUPINGS[self.group_by.value]
        groups = {}
        for label in self.labels:
            a = self.attrs[label]
            key = keyfn(a)
            if a["prefix"]:
                key = f"{a['prefix']} · {key}"
            groups.setdefault(key, []).append(label)
        return groups

    def _grid(self, members):
        """Run rows laid out in a STYLE['runs_per_row']-column grid."""
        n = max(1, int(STYLE.get("runs_per_row", 2)))
        return w.GridBox(
            [self.row_boxes[m] for m in members],
            layout=w.Layout(grid_template_columns=f"repeat({n}, max-content)",
                            grid_gap="2px 32px"))

    def _regroup(self):
        """Rebuild the run list under the selected grouping. The per-run
        widgets are reused, so ticks/labels/colours are untouched."""
        for label, handler in self._child_handlers.items():
            self.rows[label][0].unobserve(handler, names="value")
        self._child_handlers = {}

        if self.group_by.value == "none":
            ordered = [n for members in self._current_groups().values()
                       for n in members]
            self.runs_box.children = (self._grid(ordered),)
            return

        children = []
        for gname, members in self._current_groups().items():
            gtick = w.Checkbox(
                value=all(self.rows[m][0].value for m in members),
                indent=False, layout=w.Layout(width="24px"))
            gtick.observe(
                lambda ch, ms=members: self._on_group(ch, ms), names="value")
            children.append(w.HBox([gtick, w.HTML(
                f"<b>{gname}</b>&nbsp;<span style='color:#888'>"
                f"({len(members)} runs)</span>")]))
            for m in members:
                handler = (lambda ch, gt=gtick, ms=members:
                           self._on_child(gt, ms))
                self.rows[m][0].observe(handler, names="value")
                self._child_handlers[m] = handler
            children.append(self._grid(members))
        self.runs_box.children = tuple(children)

    def _on_group(self, change, members):
        """Group tickbox -> (un)tick every run in the group."""
        if self._guard:
            return
        self._guard = True
        for m in members:
            self.rows[m][0].value = change["new"]
        self._guard = False

    def _on_child(self, gtick, members):
        """Run tickbox -> keep the group tickbox in sync (all ticked?)."""
        if self._guard:
            return
        self._guard = True
        gtick.value = all(self.rows[m][0].value for m in members)
        self._guard = False

    def _colour_by_group(self):
        """Base colour per group (tab10), shading towards white within it."""
        self._color_guard = True  # Prevent saving JSON on every individual update
        base = plt.get_cmap("tab10")
        for gi, members in enumerate(self._current_groups().values()):
            b = np.array(mcolors.to_rgb(base(gi % 10)))
            for ri, m in enumerate(members):
                t = 0.55 * ri / max(len(members) - 1, 1)
                new_color = mcolors.to_hex(b * (1 - t) + t)
                self.rows[m][2].value = new_color
                self.color_cache[_cache_key(m)] = new_color
        self._save_color_cache()  # Save once at the end
        self._color_guard = False

    # --- data extraction ---------------------------------------------------
    def _get_values(self, label, source):
        """Return (values, weights_or_None, ref_scalar_or_None)."""
        if source.startswith("output_dist"):
            metric = source.split(": ")[1]
            e = self.output_dist.get(label)
            if e is None:
                return None
            vals = np.asarray(e["distributions"][metric], float)
            ref = e.get("greedy_ref", {}).get(metric)
            return vals, None, ref
        e = self.entropy.get(label)
        if e is None:
            return None
        gens = e["per_generation"]
        vals = np.array([g["mean_entropy"] for g in gens], float)
        if "token-weighted" in source:
            wts = np.array([g["num_tokens"] for g in gens], float)
            return vals, wts, e.get("token_mean_entropy_nats")
        return vals, None, e.get("per_generation_mean_nats")

    # --- plotting ----------------------------------------------------------
    def _plot(self):
        source = self.source.value
        ordered = [n for members in self._current_groups().values()
                   for n in members]
        picked = [(l, self.rows[l][1].value, self.rows[l][2].value,
                   self.rows[l][3].value.strip())
                  for l in ordered if self.rows[l][0].value]
        with self.out:
            self.out.clear_output(wait=True)
            if not picked:
                print("No runs ticked."); return
            data, missing = {}, []
            for label, disp, color, stat in picked:
                got = self._get_values(label, source)
                if got is None:
                    missing.append(label)
                else:
                    data[label] = (disp, color, stat, got)
            if missing:
                print(f"not in this file, skipped: {missing}")
            if not data:
                return

            if source == "output_dist: avg_logprob":
                print("Metric: avg log p")
                mult = -1
            else:
                mult = 1
            
            all_vals = np.concatenate([g[3][0]*mult  for g in data.values()])
            lo, hi = np.percentile(all_vals, [self.clip_lo.value, self.clip_hi.value])
            pad = 0.05 * (hi - lo) or 0.5
            x0, x1 = _as_float(self.xmin.value, lo - pad), \
                     _as_float(self.xmax.value, hi + pad)
            if x1 <= x0:
                print(f"x min ({x0}) >= x max ({x1}) — using the auto range")
                x0, x1 = lo - pad, hi + pad
            grid = np.linspace(x0, x1, 400)

            plt.rc("font", size=STYLE["font_size"])
            fig, ax = plt.subplots(figsize=STYLE["figsize"])
            handles, leg_labels, names = [], [], []
            peak_stats, all_dens = [], []
            for label, (disp, color, stat, (vals, wts, ref)) in data.items():
                vals = vals*mult
                if stat and self.stat_mode.value == "in legend":
                    disp = f"{disp}  ({stat})"
                kde = gaussian_kde(vals, bw_method=self.bw.value, weights=wts)
                dens = kde(grid)
                (line,) = ax.plot(grid, dens, label=disp, color=color,
                                  lw=STYLE["linewidth"])
                all_dens.append(dens)
                if stat and self.stat_mode.value == "at curve peak":
                    peak_stats.append((stat, dens, color))
                handles.append(line)
                leg_labels.append(disp)
                names.append(label)
                if self.show_ref.value and ref is not None and x0 <= ref <= x1:
                    ax.axvline(ref, color=color, ls="--", lw=1, alpha=0.5)
                    print(f"{label}: {ref}")
            ax.set_xlim(x0, x1)
            auto_x = source.split(": ")[1] if ":" in source else source
            ax.set_xlabel(self.xlabel.value or auto_x)
            ax.set_ylabel(self.ylabel.value)
            if self.title.value:
                ax.set_title(self.title.value)
            style_axes(ax)
            tokens = [t.strip() for t in self.legend_order.value.split(",")
                      if t.strip()]
            handles, leg_labels = order_legend(handles, leg_labels, names,
                                               tokens)
            leg = ax.legend(handles, leg_labels,
                            fontsize=STYLE["legend_fontsize"],
                            ncol=len(data), loc="lower center",
                            bbox_to_anchor=(0.5, 1.01))
            style_legend(leg)
            fig.tight_layout()
            annotate_peak_stats(ax, grid, peak_stats, all_dens)
            self._fig = fig
            plt.show()

    def _save(self):
        with self.out:
            if self._fig is None:
                print("Plot first, then save.")
                return
            self._fig.savefig(self.fname.value, dpi=STYLE["dpi"],
                              bbox_inches="tight")
            print(f"saved -> {self.fname.value}")


In [12]:
# from plot_output_dist import load_results
# from plot_entropy_dist import load_entropy_results
# from figure_gui import FigureGUI

gui = FigureGUI(
    output_dist=load_results("results/results_output_dist.json"), 
    # entropy=load_entropy_results("results/results_entropy_gsm8k.json"),
) 
gui.show() 
